In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Preprocessing & Feature Selection
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score, 
                           roc_auc_score, precision_recall_curve, average_precision_score,
                           f1_score)

# Random Forest and sampling
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from imblearn.ensemble import BalancedRandomForestClassifier

# Load and preprocess data
print("Loading data...")
data = pd.read_csv('Transactions.csv')

# Datetime feature engineering
data['trans_date_trans_time'] = pd.to_datetime(data['trans_date_trans_time'])

# Sort by datetime to ensure monotonicity
data = data.sort_values('trans_date_trans_time').reset_index(drop=True)

data['trans_hour'] = data['trans_date_trans_time'].dt.hour
data['trans_day'] = data['trans_date_trans_time'].dt.dayofweek
data['trans_month'] = data['trans_date_trans_time'].dt.month
data['trans_year'] = data['trans_date_trans_time'].dt.year
data['trans_date'] = data['trans_date_trans_time'].dt.to_period('M').dt.to_timestamp()

# Age calculation
data['dob'] = pd.to_datetime(data['dob'])
data['age'] = np.round((data['trans_date'] - data['dob']).dt.days / 365.25)

# Temporal features
data['is_weekend'] = data['trans_day'].isin([5, 6])
data['is_night'] = data['trans_hour'].between(20, 23) | data['trans_hour'].between(0, 4)
data['is_morning_rush'] = data['trans_hour'].between(7, 9)
data['is_lunch_time'] = data['trans_hour'].between(11, 13)
data['afternoon'] = data['trans_hour'].between(12, 17)
data['late_night'] = data['trans_hour'].between(0, 5)

# Transaction frequency features
print("Creating transaction frequency features...")

# Time since last transaction
data = data.sort_values(['cc_num', 'trans_date_trans_time'])
data['time_since_last_tx'] = data.groupby('cc_num')['trans_date_trans_time'].diff().dt.total_seconds() / 3600
data['time_since_last_tx'] = data['time_since_last_tx'].fillna(24)

# Daily transaction pattern
data['day_of_month'] = data['trans_date_trans_time'].dt.day
data['is_month_end'] = data['day_of_month'] >= 25

# Amount statistics per card
card_stats = data.groupby('cc_num')['amt'].agg(['mean', 'std']).reset_index()
card_stats.columns = ['cc_num', 'card_amt_mean', 'card_amt_std']
data = data.merge(card_stats, on='cc_num', how='left')

data['amt_to_card_mean_ratio'] = data['amt'] / data['card_amt_mean']
data['amt_z_score'] = (data['amt'] - data['card_amt_mean']) / data['card_amt_std']
data['amt_z_score'] = data['amt_z_score'].fillna(0)

# Amount-based features
data['amt_log'] = np.log1p(data['amt'])
data['is_high_amount'] = data['amt'] > data['amt'].quantile(0.95)
data['is_low_amount'] = data['amt'] < data['amt'].quantile(0.05)

# Distance features
data['dist_merch2cust'] = np.sqrt(
    (data['lat'] - data['merch_lat'])**2 + 
    (data['long'] - data['merch_long'])**2
)
data['is_long_distance'] = data['dist_merch2cust'] > data['dist_merch2cust'].quantile(0.95)

# Drop unnecessary columns
cols_to_drop = ['trans_num', 'trans_date_trans_time', 'trans_date', 'dob', 'day_of_month', 
                'card_amt_mean', 'card_amt_std']
cols_to_drop = [col for col in cols_to_drop if col in data.columns]
data.drop(cols_to_drop, axis=1, inplace=True)

target = 'is_fraud'
X = data.drop(columns=[target])
y = data[target]

print(f"Dataset shape: {X.shape}")
print(f"🎯 Fraud rate: {y.mean():.4f} ({y.sum()} fraud cases)")

# Encode categorical features
print("Encoding categorical features...")
cat_cols = X.select_dtypes(include='object').columns
for col in cat_cols:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

# Encode boolean features
bool_cols = X.select_dtypes(include='bool').columns
for col in bool_cols:
    X[col] = X[col].astype(int)

print(f"Final feature set: {X.shape[1]} features")

# Train-test-holdout split
print("Creating train-test-holdout split...")
X_temp, X_hold, y_temp, y_hold = train_test_split(
    X, y, test_size=0.003, stratify=y, random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X_temp, y_temp, test_size=0.2, stratify=y_temp, random_state=42
)

print(f"Train set: {X_train.shape}, Fraud rate: {y_train.mean():.4f}")
print(f"Test set: {X_test.shape}, Fraud rate: {y_test.mean():.4f}")
print(f"Holdout set: {X_hold.shape}, Fraud rate: {y_hold.mean():.4f}")

# 🔍 Feature Selection for Random Forest
print("\n🔍 Performing feature selection...")

# Variance Threshold
vt = VarianceThreshold(threshold=0.01)
vt.fit(X_train)
selected_vt_features = X_train.columns[vt.get_support()].tolist()

if len(selected_vt_features) == 0:
    print("Variance Threshold removed all features! Using all features instead.")
    selected_vt_features = X_train.columns.tolist()

print(f"Variance Threshold selected {len(selected_vt_features)} features")

# Mutual Information for feature ranking
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train[selected_vt_features])
mi_scores = mutual_info_classif(X_train_scaled, y_train, random_state=42)

mi_df = pd.DataFrame({
    'Feature': selected_vt_features, 
    'Mutual_Info': mi_scores
}).sort_values(by='Mutual_Info', ascending=False)

# Random Forest Feature Importance
rf_base = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_base.fit(X_train[selected_vt_features], y_train)
rf_importance = rf_base.feature_importances_

rf_df = pd.DataFrame({
    'Feature': selected_vt_features, 
    'RF_Importance': rf_importance
}).sort_values(by='RF_Importance', ascending=False)

# Combine scores
score_df = mi_df.merge(rf_df, on='Feature')
score_df['Composite_Score'] = score_df[['Mutual_Info', 'RF_Importance']].mean(axis=1)
score_df = score_df.sort_values(by='Composite_Score', ascending=False)

print("\n🏆 Top 20 Features for Random Forest:")
print(score_df.head(20)[['Feature', 'Composite_Score']])

# Final feature selection
top_n = min(30, len(score_df))  # Select top 30 features
top_features = score_df.head(top_n)['Feature'].tolist()
print(f"Selected {len(top_features)} top features for Random Forest")

# Apply feature selection
X_train_rf = X_train[top_features]
X_test_rf = X_test[top_features]
X_hold_rf = X_hold[top_features]

# Handle Class Imbalance with SMOTE
print("\nApplying SMOTE for class imbalance...")
smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X_train_rf, y_train)

print(f"Original train: {X_train_rf.shape}, Fraud: {y_train.sum()}")
print(f"After SMOTE: {X_smote.shape}, Fraud: {y_smote.sum()}")



📥 Loading data...
💳 Creating transaction frequency features...
📊 Dataset shape: (1296675, 41)
🎯 Fraud rate: 0.0058 (7506 fraud cases)
🔤 Encoding categorical features...
✅ Final feature set: 41 features
📊 Creating train-test-holdout split...
Train set: (1034227, 41), Fraud rate: 0.0058
Test set: (258557, 41), Fraud rate: 0.0058
Holdout set: (3891, 41), Fraud rate: 0.0059

🔍 Performing feature selection...
📈 Variance Threshold selected 37 features

🏆 Top 20 Features for Random Forest:
                   Feature  Composite_Score
12                 amt_log         0.129479
11                     amt         0.085577
0                 is_night         0.071887
13          is_high_amount         0.063880
15  amt_to_card_mean_ratio         0.051839
16             amt_z_score         0.039389
1               is_weekend         0.037845
10              trans_hour         0.036683
9                 category         0.035073
2                afternoon         0.034455
3               trans_year  

In [ ]:
#  Random Forest Hyperparameter Tuning
print("\nTraining Random Forest with optimized parameters...")

# Define Random Forest parameters for fraud detection
rf_params = {
    'n_estimators': 200,
    'max_depth': 15,
    'min_samples_split': 10,
    'min_samples_leaf': 4,
    'max_features': 'sqrt',
    'bootstrap': True,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': 0
}

# Enhanced Evaluation Function for Random Forest
def evaluate_rf_model(clf, X_train, y_train, X_test, y_test, model_name, use_smote=False):
    """Evaluate Random Forest model with comprehensive metrics"""
    
    # Train on SMOTE data if specified
    if use_smote:
        X_train_use, y_train_use = X_smote, y_smote
    else:
        X_train_use, y_train_use = X_train, y_train
    
    # Train model
    clf.fit(X_train_use, y_train_use)
    
    # Predictions
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]
    
    # Comprehensive metrics
    report = classification_report(y_test, y_pred, output_dict=True)
    roc_auc = roc_auc_score(y_test, y_proba)
    avg_precision = average_precision_score(y_test, y_proba)
    
    # Fraud class metrics
    fraud_report = report.get('1', {'precision': 0, 'recall': 0, 'f1-score': 0})
    fraud_f1 = f1_score(y_test, y_pred, pos_label=1)
    
    # Geometric Mean
    g_mean = np.sqrt(fraud_report['recall'] * report['0']['recall'])
    
    return {
        'Model': model_name,
        'ROC_AUC': roc_auc,
        'Average_Precision': avg_precision,
        'Fraud_Precision': fraud_report['precision'],
        'Fraud_Recall': fraud_report['recall'],
        'Fraud_F1': fraud_f1,
        'G_Mean': g_mean,
        'Accuracy': accuracy_score(y_test, y_pred),
        'y_proba': y_proba,
        'Model_Object': clf,
        'Feature_Importance': clf.feature_importances_
    }

# Find Optimal Threshold for Fraud Detection
def find_optimal_threshold(y_true, y_proba):
    """Find optimal threshold based on F1 score for fraud class"""
    thresholds = np.arange(0.05, 0.95, 0.02)
    best_threshold = 0.5
    best_f1 = 0
    
    for threshold in thresholds:
        y_pred = (y_proba >= threshold).astype(int)
        f1 = f1_score(y_true, y_pred, pos_label=1)
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold
    
    return best_threshold



🧠 Training Random Forest with optimized parameters...


In [3]:

# Train and Evaluate Random Forest Variants
print("\nTraining Random Forest variants...")

rf_results = []

# Train different Random Forest configurations
rf_configs = [
    {
        'name': 'RF Balanced',
        'params': {**rf_params, **{'class_weight': 'balanced'}},
        'use_smote': False
    },
    {
        'name': 'RF SMOTE', 
        'params': {**rf_params, **{'class_weight': 'balanced'}},
        'use_smote': True
    },
    {
        'name': 'RF Custom Weight',
        'params': {**rf_params, **{'class_weight': {0: 1, 1: 10}}},  # Higher weight for fraud
        'use_smote': False
    },
    {
        'name': 'Balanced RF',
        'model': BalancedRandomForestClassifier(
            n_estimators=200,
            max_depth=15,
            min_samples_split=10,
            min_samples_leaf=4,
            max_features='sqrt',
            sampling_strategy='auto',
            random_state=42,
            n_jobs=-1
        ),
        'use_smote': False,
        'is_balanced_rf': True
    },
    {
        'name': 'RF No Weight',
        'params': {**rf_params, **{'class_weight': None}},
        'use_smote': False
    }
]

for config in rf_configs:
    try:
        print(f"Training {config['name']}...")
        
        if config.get('is_balanced_rf', False):
            clf = config['model']
        else:
            clf = RandomForestClassifier(**config['params'])
        
        result = evaluate_rf_model(
            clf, X_train_rf, y_train, X_test_rf, y_test, 
            config['name'], use_smote=config['use_smote']
        )
        rf_results.append(result)
        
        print(f"✅ {config['name']}: "
              f"AP={result['Average_Precision']:.4f}, "
              f"Fraud F1={result['Fraud_F1']:.4f}")
              
    except Exception as e:
        print(f"{config['name']} failed: {e}")
        # Create a simple model as fallback
        try:
            print(f"Trying fallback for {config['name']}...")
            clf_simple = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
            clf_simple.fit(X_train_rf, y_train)
            
            y_pred = clf_simple.predict(X_test_rf)
            y_proba = clf_simple.predict_proba(X_test_rf)[:, 1]
            
            report = classification_report(y_test, y_pred, output_dict=True)
            fraud_report = report.get('1', {'precision': 0, 'recall': 0, 'f1-score': 0})
            
            fallback_result = {
                'Model': config['name'] + ' (Fallback)',
                'ROC_AUC': roc_auc_score(y_test, y_proba),
                'Average_Precision': average_precision_score(y_test, y_proba),
                'Fraud_Precision': fraud_report['precision'],
                'Fraud_Recall': fraud_report['recall'], 
                'Fraud_F1': f1_score(y_test, y_pred, pos_label=1),
                'G_Mean': np.sqrt(fraud_report['recall'] * report['0']['recall']),
                'Accuracy': accuracy_score(y_test, y_pred),
                'y_proba': y_proba,
                'Model_Object': clf_simple,
                'Feature_Importance': clf_simple.feature_importances_
            }
            rf_results.append(fallback_result)
            print(f"{config['name']} (Fallback): AP={fallback_result['Average_Precision']:.4f}")
            
        except Exception as e2:
            print(f"Fallback also failed for {config['name']}: {e2}")

# Check if we have any results
if not rf_results:
    print("All models failed! Training a simple default model...")
    # Train a simple default model
    default_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    default_model.fit(X_train_rf, y_train)
    
    y_pred = default_model.predict(X_test_rf)
    y_proba = default_model.predict_proba(X_test_rf)[:, 1]
    
    report = classification_report(y_test, y_pred, output_dict=True)
    fraud_report = report.get('1', {'precision': 0, 'recall': 0, 'f1-score': 0})
    
    default_result = {
        'Model': 'RF Default',
        'ROC_AUC': roc_auc_score(y_test, y_proba),
        'Average_Precision': average_precision_score(y_test, y_proba),
        'Fraud_Precision': fraud_report['precision'],
        'Fraud_Recall': fraud_report['recall'],
        'Fraud_F1': f1_score(y_test, y_pred, pos_label=1),
        'G_Mean': np.sqrt(fraud_report['recall'] * report['0']['recall']),
        'Accuracy': accuracy_score(y_test, y_pred),
        'y_proba': y_proba,
        'Model_Object': default_model,
        'Feature_Importance': default_model.feature_importances_
    }
    rf_results.append(default_result)

# Select Best Random Forest Model
rf_results_df = pd.DataFrame(rf_results)

print("\n" + "="*80)
print("🏆 RANDOM FOREST MODEL COMPARISON")
print("="*80)

if not rf_results_df.empty:
    rf_results_df = rf_results_df.sort_values('Average_Precision', ascending=False)
    print(rf_results_df[['Model', 'Average_Precision', 'Fraud_F1', 'Fraud_Recall', 'ROC_AUC']].round(4))

    # Select Best Model
    best_rf_result = rf_results_df.iloc[0]
    best_rf_model = best_rf_result['Model_Object']
    best_rf_name = best_rf_result['Model']

    print(f"\n🏆 BEST RANDOM FOREST MODEL: {best_rf_name}")

    # Threshold Optimization
    y_test_proba = best_rf_result['y_proba']
    optimal_threshold = find_optimal_threshold(y_test, y_test_proba)
    print(f"Optimal threshold: {optimal_threshold:.3f}")

    # Final Evaluation with Optimal Threshold
    y_test_pred_optimal = (y_test_proba >= optimal_threshold).astype(int)

    print("\n" + "="*80)
    print("FINAL RANDOM FOREST PERFORMANCE (Test Set)")
    print("="*80)
    print(f"Model: {best_rf_name}")
    print(f"Optimal Threshold: {optimal_threshold:.3f}")

    print("\nClassification Report with Optimal Threshold:")
    print(classification_report(y_test, y_test_pred_optimal))

    conf_matrix = confusion_matrix(y_test, y_test_pred_optimal)
    print("Confusion Matrix:")
    print(conf_matrix)

    print(f"ROC AUC: {roc_auc_score(y_test, y_test_proba):.4f}")
    print(f"Average Precision: {average_precision_score(y_test, y_test_proba):.4f}")

    # 🔧 Retrain Best Model on Full Training Data
    print(f"\n🔧 Retraining best Random Forest model on full training data...")

    # Determine if we should use SMOTE based on best model
    use_smote_final = 'SMOTE' in best_rf_name

    if use_smote_final:
        X_final_train, y_final_train = smote.fit_resample(X_temp[top_features], y_temp)
    else:
        X_final_train, y_final_train = X_temp[top_features], y_temp

    # Use the same parameters as the best model
    if hasattr(best_rf_model, 'get_params'):
        final_params = best_rf_model.get_params()
        final_model = RandomForestClassifier(**final_params)
    else:
        # For BalancedRandomForest or other special models
        final_model = best_rf_model.__class__(**best_rf_model.get_params())
    
    final_model.fit(X_final_train, y_final_train)

    print(f"✅ Final model trained on {len(X_final_train)} samples")

    # Holdout Set Evaluation
    print("\n" + "="*80)
    print(" HOLDOUT SET EVALUATION")
    print("="*80)

    y_hold_proba = final_model.predict_proba(X_hold_rf)[:, 1]
    y_hold_pred = (y_hold_proba >= optimal_threshold).astype(int)

    print(" Classification Report (Holdout Set):")
    print(classification_report(y_hold, y_hold_pred))

    holdout_conf_matrix = confusion_matrix(y_hold, y_hold_pred)
    print(" Confusion Matrix (Holdout):")
    print(holdout_conf_matrix)

    holdout_ap = average_precision_score(y_hold, y_hold_proba)
    holdout_auc = roc_auc_score(y_hold, y_hold_proba)
    print(f" Holdout Average Precision: {holdout_ap:.4f}")
    print(f" Holdout ROC AUC: {holdout_auc:.4f}")

    # Save Random Forest Model and Results
    print("\nSaving Random Forest model and results...")

    import joblib

    # Save final model
    joblib.dump(final_model, 'rf_fraud_model.pkl')
    joblib.dump(top_features, 'rf_selected_features.pkl')

    # Save datasets
    holdout_set = pd.concat([X_hold_rf, y_hold], axis=1)
    holdout_set.to_csv('rf_holdout_set.csv', index=False)

    training_set = pd.concat([X_temp[top_features], y_temp], axis=1)
    training_set.to_csv('rf_training_set.csv', index=False)

    # Save predictions
    predictions_df = pd.DataFrame({
        'actual': y_test,
        'predicted_prob': y_test_proba,
        'predicted_class_05': (y_test_proba >= 0.5).astype(int),
        'predicted_class_optimal': y_test_pred_optimal
    })
    predictions_df.to_csv('rf_predictions.csv', index=False)

    # Save feature importance
    feature_imp_df = pd.DataFrame({
        'feature': top_features,
        'importance': final_model.feature_importances_
    }).sort_values('importance', ascending=False)
    feature_imp_df.to_csv('rf_feature_importance.csv', index=False)

    # Save model results
    rf_results_df.to_csv('rf_model_results.csv', index=False)

    print("\nRandom Forest training completed successfully!")
    print(f"Best Model: {best_rf_name}")
    print(f"Test Average Precision: {best_rf_result['Average_Precision']:.4f}")
    print(f"Holdout Average Precision: {holdout_ap:.4f}")
    print(f"Optimal Threshold: {optimal_threshold:.3f}")

    # Random Forest Feature Importance Visualization
    plt.figure(figsize=(12, 10))
    top_20_features = feature_imp_df.head(20)
    plt.barh(top_20_features['feature'], top_20_features['importance'])
    plt.title('Random Forest Feature Importance - Top 20 Features')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.savefig('rf_feature_importance.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("📊 Feature importance plot saved!")

    # Precision-Recall Curve
    plt.figure(figsize=(10, 8))
    precision, recall, _ = precision_recall_curve(y_test, y_test_proba)
    plt.plot(recall, precision, marker='.')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Random Forest Precision-Recall Curve')
    plt.grid(True)
    plt.savefig('rf_precision_recall_curve.png', dpi=300, bbox_inches='tight')
    plt.close()

    print("\n Saved files:")
    print("   - rf_fraud_model.pkl (Final trained model)")
    print("   - rf_selected_features.pkl (Feature list)")
    print("   - rf_holdout_set.csv")
    print("   - rf_training_set.csv")
    print("   - rf_predictions.csv")
    print("   - rf_feature_importance.csv")
    print("   - rf_model_results.csv")
    print("   - rf_feature_importance.png")
    print("   - rf_precision_recall_curve.png")

    # Business Impact Analysis
    print("\n" + "="*80)
    print(" BUSINESS IMPACT ANALYSIS")
    print("="*80)

    # Calculate cost savings
    fraud_prevented = conf_matrix[1, 1]  # True Positives
    false_positives = conf_matrix[0, 1]  # False Positives

    if len(data[data['is_fraud'] == 1]) > 0:
        avg_fraud_amount = data[data['is_fraud'] == 1]['amt'].mean()
    else:
        avg_fraud_amount = 100  # Fallback average

    false_positive_cost = 10  # Estimated cost per false positive in dollars

    total_savings = fraud_prevented * avg_fraud_amount
    total_fp_cost = false_positives * false_positive_cost
    net_savings = total_savings - total_fp_cost

    print(f"Estimated Fraud Prevented: {fraud_prevented} transactions")
    print(f"Average Fraud Amount: ${avg_fraud_amount:.2f}")
    print(f"Total Fraud Savings: ${total_savings:.2f}")
    print(f"False Positives: {false_positives} transactions")
    print(f"False Positive Cost: ${total_fp_cost:.2f}")
    print(f"NET SAVINGS: ${net_savings:.2f}")

    # Model Inference Function
    def predict_fraud(model, features, transaction_data, threshold=0.5):
        """Helper function for making fraud predictions"""
        # Ensure we have the right features
        missing_features = set(features) - set(transaction_data.columns)
        if missing_features:
            raise ValueError(f"Missing features: {missing_features}")
        
        # Select and order features correctly
        X_pred = transaction_data[features]
        
        # Predict probabilities
        fraud_proba = model.predict_proba(X_pred)[:, 1]
        predictions = (fraud_proba >= threshold).astype(int)
        
        return predictions, fraud_proba

    print(f"\n Model ready for deployment!")
    print(f" Use threshold: {optimal_threshold:.3f} for optimal performance")
    print(f"Key metric: Average Precision = {holdout_ap:.4f}")

else:
    print("No models were successfully trained!")

print("\n Random Forest training process completed!")


📈 Training Random Forest variants...
Training RF Balanced...
✅ RF Balanced: AP=0.8367, Fraud F1=0.5541
Training RF SMOTE...
✅ RF SMOTE: AP=0.7390, Fraud F1=0.4707
Training RF Custom Weight...
✅ RF Custom Weight: AP=0.8861, Fraud F1=0.8281
Training Balanced RF...
✅ Balanced RF: AP=0.8127, Fraud F1=0.2668
Training RF No Weight...
✅ RF No Weight: AP=0.8850, Fraud F1=0.7992

🏆 RANDOM FOREST MODEL COMPARISON
              Model  Average_Precision  Fraud_F1  Fraud_Recall  ROC_AUC
2  RF Custom Weight             0.8861    0.8281        0.7916   0.9954
4      RF No Weight             0.8850    0.7992        0.6754   0.9948
0       RF Balanced             0.8367    0.5541        0.8931   0.9955
3       Balanced RF             0.8127    0.2668        0.9633   0.9948
1          RF SMOTE             0.7390    0.4707        0.8784   0.9868

🏆 BEST RANDOM FOREST MODEL: RF Custom Weight
🎯 Optimal threshold: 0.510

🎯 FINAL RANDOM FOREST PERFORMANCE (Test Set)
Model: RF Custom Weight
Optimal Threshold

In [ ]:
import shap
import matplotlib.pyplot as plt
import seaborn as sns

# 🎯 SHAP Analysis for Random Forest
print("\n" + "="*80)
print("🔍 SHAP ANALYSIS FOR RANDOM FOREST")
print("="*80)

# Initialize SHAP explainer
print("Initializing SHAP explainer...")
explainer = shap.TreeExplainer(final_model)

# Calculate SHAP values
print("Calculating SHAP values...")
X_explain = X_test_rf  # Use test set for explanation
shap_values = explainer.shap_values(X_explain)

# For binary classification, we get shap_values[1] for the positive class (fraud)
if isinstance(shap_values, list):
    shap_values_fraud = shap_values[1]  # Fraud class (class 1)
else:
    shap_values_fraud = shap_values

print(f"SHAP values shape: {shap_values_fraud.shape}")

# 📊 1. SHAP Summary Plot
print("Creating SHAP summary plot...")
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values_fraud, X_explain, show=False)
plt.title("SHAP Summary Plot - Random Forest\n(Impact on Fraud Prediction)", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('rf_shap_summary.png', dpi=300, bbox_inches='tight')
plt.close()
print("✅ SHAP summary plot saved!")

# 📊 2. SHAP Bar Plot (Mean Absolute Impact)
print("Creating SHAP bar plot...")
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values_fraud, X_explain, plot_type="bar", show=False)
plt.title("SHAP Feature Importance - Random Forest\n(Mean |SHAP| Impact)", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('rf_shap_bar.png', dpi=300, bbox_inches='tight')
plt.close()
print("✅ SHAP bar plot saved!")

# 📊 3. SHAP Waterfall Plot for Specific Examples
print("Creating SHAP waterfall plots...")
# Plot top 3 fraud cases and top 3 legitimate cases
fraud_indices = y_test[y_test == 1].index[:3]
non_fraud_indices = y_test[y_test == 0].index[:3]

for i, idx in enumerate(fraud_indices):
    plt.figure(figsize=(12, 8))
    shap.waterfall_plot(
        shap.Explanation(
            values=shap_values_fraud[idx],
            base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value,
            data=X_explain.iloc[idx],
            feature_names=X_explain.columns.tolist()
        ),
        show=False
    )
    plt.title(f"SHAP Waterfall Plot - Fraud Case {i+1}\n(Predicted Probability: {final_model.predict_proba(X_explain.iloc[idx:idx+1])[0,1]:.3f})", 
              fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'rf_shap_waterfall_fraud_{i+1}.png', dpi=300, bbox_inches='tight')
    plt.close()

for i, idx in enumerate(non_fraud_indices):
    plt.figure(figsize=(12, 8))
    shap.waterfall_plot(
        shap.Explanation(
            values=shap_values_fraud[idx],
            base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value,
            data=X_explain.iloc[idx],
            feature_names=X_explain.columns.tolist()
        ),
        show=False
    )
    plt.title(f"SHAP Waterfall Plot - Legitimate Case {i+1}\n(Predicted Probability: {final_model.predict_proba(X_explain.iloc[idx:idx+1])[0,1]:.3f})", 
              fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'rf_shap_waterfall_legit_{i+1}.png', dpi=300, bbox_inches='tight')
    plt.close()
print("✅ SHAP waterfall plots saved!")

# 📊 4. SHAP Dependence Plots for Top Features
print("Creating SHAP dependence plots...")
top_features = feature_imp_df.head(5)['feature'].tolist()

for feature in top_features:
    plt.figure(figsize=(10, 6))
    shap.dependence_plot(
        feature, 
        shap_values_fraud, 
        X_explain, 
        show=False,
        interaction_index=None
    )
    plt.title(f"SHAP Dependence Plot - {feature}", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'rf_shap_dependence_{feature}.png', dpi=300, bbox_inches='tight')
    plt.close()
print("✅ SHAP dependence plots saved!")

# 📊 5. SHAP Force Plot for Individual Predictions
print("Creating SHAP force plots...")
# Create force plots for the same examples
for i, idx in enumerate(fraud_indices):
    plt.figure(figsize=(12, 4))
    shap.force_plot(
        explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value,
        shap_values_fraud[idx],
        X_explain.iloc[idx],
        matplotlib=True,
        show=False
    )
    plt.title(f"SHAP Force Plot - Fraud Case {i+1}", fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'rf_shap_force_fraud_{i+1}.png', dpi=300, bbox_inches='tight')
    plt.close()

print("✅ SHAP force plots saved!")

# 📊 6. SHAP Feature Importance Comparison
print("Creating SHAP vs Traditional feature importance comparison...")
# Compare SHAP importance with traditional feature importance
shap_importance = pd.DataFrame({
    'feature': X_explain.columns,
    'shap_importance': np.abs(shap_values_fraud).mean(axis=0)
}).sort_values('shap_importance', ascending=False)

# Merge with traditional feature importance
comparison_df = feature_imp_df.merge(shap_importance, on='feature')
comparison_df = comparison_df.sort_values('importance', ascending=False)

# Plot comparison
plt.figure(figsize=(14, 10))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# Traditional Feature Importance
ax1.barh(comparison_df['feature'].head(15), comparison_df['importance'].head(15))
ax1.set_title('Traditional Feature Importance\n(Random Forest)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Importance')

# SHAP Feature Importance
ax2.barh(comparison_df['feature'].head(15), comparison_df['shap_importance'].head(15))
ax2.set_title('SHAP Feature Importance\n(Mean |SHAP| Value)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Mean |SHAP| Impact')

plt.tight_layout()
plt.savefig('rf_shap_vs_traditional_importance.png', dpi=300, bbox_inches='tight')
plt.close()
print("✅ SHAP vs traditional importance comparison saved!")

# 📊 7. SHAP Decision Plot
print("Creating SHAP decision plot...")
plt.figure(figsize=(12, 8))
shap.decision_plot(
    explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value,
    shap_values_fraud[:50],  # First 50 examples
    X_explain.iloc[:50],
    feature_names=list(X_explain.columns),
    show=False
)
plt.title("SHAP Decision Plot - Random Forest\n(First 50 Transactions)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('rf_shap_decision_plot.png', dpi=300, bbox_inches='tight')
plt.close()
print("✅ SHAP decision plot saved!")

# 💾 Save SHAP Analysis Results
print("\n💾 Saving SHAP analysis results...")

# Save SHAP values and explanation data
shap_data = {
    'shap_values': shap_values_fraud,
    'expected_value': explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value,
    'feature_names': X_explain.columns.tolist(),
    'X_values': X_explain.values
}

import joblib
joblib.dump(shap_data, 'rf_shap_analysis_data.pkl')

# Save SHAP importance dataframe
shap_importance.to_csv('rf_shap_feature_importance.csv', index=False)

print("\n📊 SHAP Analysis Summary:")
print(f"• Expected Value (Base Rate): {shap_data['expected_value']:.4f}")
print(f"• Top 5 Most Important Features by SHAP:")
for i, row in shap_importance.head().iterrows():
    print(f"  {i+1}. {row['feature']}: {row['shap_importance']:.4f}")

print(f"\n📁 SHAP Analysis Files Saved:")
print("   - rf_shap_summary.png (Overall feature impact)")
print("   - rf_shap_bar.png (Feature importance ranking)")
print("   - rf_shap_waterfall_*.png (Individual case explanations)")
print("   - rf_shap_dependence_*.png (Feature relationship plots)")
print("   - rf_shap_force_*.png (Force plots for individual predictions)")
print("   - rf_shap_vs_traditional_importance.png (Comparison)")
print("   - rf_shap_decision_plot.png (Decision paths)")
print("   - rf_shap_analysis_data.pkl (Raw SHAP data)")
print("   - rf_shap_feature_importance.csv (SHAP importance scores)")

print("\n🎯 SHAP Analysis Complete!")
print("   Use these plots to explain model decisions to stakeholders")
print("   and understand what features drive fraud predictions.")